# Interlude — Python's Own Rules
## Dunders, Shadowing, and Other Things That Bite

*An unnumbered companion to the main series — dip into it whenever one of these bites you, no need to read it start to finish, and no obligation to read it in order with Notebooks 1, 2, 3...*

**Authors:** Isht Vibhu &nbsp;•&nbsp; Brijesh Shukla &nbsp;•&nbsp; Mohammad Amir &nbsp;•&nbsp; Dr. Ajay Verma

## Why this notebook exists, and why it isn't numbered

Notebooks 1 onward teach you Python *forward* — variables, then loops, then functions, each one building on the last. This one is different. It exists because Python has a handful of its own internal rules — not physics, not materials science, just *how the language itself is built* — that quietly bite almost everyone at some point, usually while they're deep in the middle of something else entirely.

So this interlude sits outside the main sequence on purpose. Read it once you've met variables, loops, and functions (Notebook 1) — you'll want that vocabulary already in hand. After that, come back whenever a numpy or materials-science notebook throws something confusing at you that *isn't* about numpy or materials science at all — it's about Python.

To keep the spotlight on the language and not the field, every example below leans on the most elementary physics I could reach for: things falling, a pendulum swinging, Ohm's law. Nothing here requires anything past school physics — the numbers are just furniture.

---
# 1. Dunders — the names Python reserves for itself

### The hand-wavy picture

Every object in Python carries a small set of labels that *Python itself* wrote, not you. They're spelled with double underscores on both sides — `__name__`, `__len__`, `__init__` — and programmers say them out loud as **"dunder"**, short for **d**ouble **under**score, because "underscore-underscore-name-underscore-underscore" is a mouthful nobody has time for.

Think of it like the difference between the name you gave your dog and the barcode the vet clinic put on his file. You're free to call him whatever you like. The barcode, you don't touch — it's how the *system* keeps track of him, and it follows a format the system defined, not you.

### A little more precisely

The double underscores are a fence, not a decoration. They mark a name as belonging to Python's own vocabulary — a hook the language uses to plug ordinary syntax (`print(x)`, `len(x)`, `x + y`) into whatever behaviour an object defines. You almost never *invent* a new dunder yourself; you either *read* one Python already defines, or, much later, *override* one to customize how your own class behaves.

You genuinely don't need to memorise many. This is close to the complete list of dunders you'll meet in ordinary, non-class-writing physics code:

| Dunder | What triggers it |
|---|---|
| `__name__` | the name of a class, function, or module |
| `__init__` | runs when an object is created, `MyClass(...)` |
| `__str__` | what `print(obj)` shows |
| `__len__` | what `len(obj)` returns |
| `__main__` | the special value of `__name__` when a script is *run directly* (you met this exact dunder in Notebook 0's script example, in `if __name__ == "__main__":`) |

In [1]:
# a free-fall calculation, nothing more -- the physics here is just furniture
g = 9.8          # acceleration due to gravity, m/s^2
t = 2.0          # time elapsed, s
v = g * t        # velocity after falling from rest, m/s

print("v =", v)
print("the TYPE of v is:", type(v))
print("and the NAME of that type is:", type(v).__name__)

v = 19.6
the TYPE of v is: <class 'float'>
and the NAME of that type is: float


Run the cell above and look closely at the last two lines. `type(v)` hands you back the *class itself* — `<class 'float'>` — a whole object, not a string. `type(v).__name__` reaches one level deeper and asks that class for its own short, human-readable name: `'float'`.

Note what *doesn't* work: there is no `type(v).name`. Only `.__name__` exists, because that's the exact label Python's own machinery defines — not a convenience you get to spell however you like.

### ⭐ Exercise I.1 (warm-up)

Below, `T` is the period of a pendulum, computed with `math.sqrt`. Print a single sentence of the form `"T is of type: float"` — but build the word `"float"` from `T` itself, using a dunder, rather than typing the word `"float"` by hand.

In [3]:
# ---- Exercise I.1: your attempt ----
import math

L = 1.0                       # pendulum length, m
g = 9.8                       # m/s^2
T = 2 * math.pi * math.sqrt(L / g)   # period of a simple pendulum, s

print("T is of type:",type(T))

T is of type: <class 'float'>


In [2]:
#@title  ▶ Solution to Exercise I.1  (click to reveal — try it yourself first!)

import math

L = 1.0
g = 9.8
T = 2 * math.pi * math.sqrt(L / g)

print("T is of type:", type(T).__name__)

T is of type: float


In [5]:
type(T)              # <class 'float'>   -- a class object
type(T).__name__     # 'float'            -- a string
type(type(T))        # <class 'type'>     -- type(T) is itself an instance of `type`
type(type(T).__name__)  # <class 'str'>   -- and .__name__ is a genuine string, usable like any other

str

You put four bare expressions in one cell, back to back, with no print().  
In a Jupyter/IPython cell, only the value of the very last line gets auto-displayed.
The first three (type(T), type(T).__name__, type(type(T))) are still evaluated — Python does the work — but  
since nothing captures or prints their results, and they're not the last line, the notebook just throws them away silently.  
nly type(type(T).__name__) survives to be shown, and its answer, str, appears below the cell exactly as you got it.

In [6]:
print(type(T))
print(type(T).__name__)
print(type(type(T)))
print(type(type(T).__name__))

<class 'float'>
float
<class 'type'>
<class 'str'>


---
# 2. Wildcard imports — borrowing tools without labelling them

### The hand-wavy picture

`import math` hands you a labelled toolbox: everything inside is reached as `math.sqrt`, `math.pi`, `math.sin` — a little verbose, but you always know where a tool came from.

`from math import *` dumps the whole toolbox onto your workbench, unlabelled, mixed in with whatever else was already lying there. Faster to grab a spanner. Much easier to reach for the wrong one.

### A little more precisely

`from math import *` (or `from numpy import *`, which the numpy notebook in this series meets and then deliberately moves away from) rebinds every name the module exports directly into your current namespace. If Python's own builtin `sum`, `abs`, `min`, or `max` happens to share a name with something the module exports, the module's version *silently replaces* the builtin — and it stays replaced for the rest of that session, no matter what you `import` afterwards.

In [4]:
# Ohm's law, done the wildcard way
from math import *      # brings sqrt, pi, sin, cos, ... straight into this notebook, unlabelled

I = 2.0     # current, A
R = 5.0     # resistance, ohm
V = I * R   # voltage, V
print("V =", V)

# so far, harmless. now watch what already happened quietly in the background:
currents = [1.0, 2.0, 3.0, 4.0]
print("total current, the way I'd normally write it:", sum(currents))

V = 10.0
total current, the way I'd normally write it: 10.0


That last line still works — `math` doesn't happen to export anything called `sum`, so Python's own builtin survived untouched this time. But that's luck, not a guarantee, and it's exactly the kind of luck that runs out. Try the next cell, which does the same thing with `numpy` instead of `math`.

In [7]:
from numpy import *   # numpy DOES export its own sum, abs, min, max ... and this quietly replaces the builtins

currents = [1.0, 2.0, 3.0, 4.0]
print("total current:", sum(currents))   # still looks completely normal

# but the SAME name 'sum' is no longer Python's builtin -- it's numpy's, permanently, for the
# rest of this session. compare:
import builtins
print("builtins.sum :", builtins.sum(currents))
print("the bare sum() you'd type now:", sum(currents))
print("are they even the same function?", sum is builtins.sum)

total current: 10.0
builtins.sum : 10.0
the bare sum() you'd type now: 10.0
are they even the same function? False


Both lines print the same *number* here — `10.0` — so nothing looks wrong. That's precisely what makes this trap dangerous: it very often produces the right answer, right up until the one time it doesn't. `numpy`'s `sum`, for instance, refuses a generator expression that the builtin `sum` accepts happily — a distinction that costs nobody anything, until the day it costs someone an entire afternoon of debugging, hunting a `TypeError` in a function that used to work.

**The habit worth keeping:** prefer `import numpy as np` and always write `np.sum`, `np.abs`, `np.array` — never the bare name. It's a few extra characters, in exchange for never once wondering, three files later, which `sum` you're actually calling.

### ⭐ Exercise I.2

Restart this notebook's kernel (`Runtime → Restart session` in Colab, or the equivalent in Jupyter) *before* attempting this, so neither wildcard import above is still active. Then, using **only** `import numpy as np`, write one line that computes the average of the `currents` list above, using numpy's mean function, correctly labelled.

In [6]:
# ---- Exercise I.2: your attempt ----
import numpy as np

currents = [1.0, 2.0, 3.0, 4.0]

# average_current = ...
# print("average current:", average_current)

In [7]:
#@title  ▶ Solution to Exercise I.2  (click to reveal — try it yourself first!)

import numpy as np

currents = [1.0, 2.0, 3.0, 4.0]
average_current = np.mean(currents)
print("average current:", average_current)

average current: 2.5


---
# 3. Mutability, and numbers that quietly change shape

### The hand-wavy picture

A Python list is a row of open boxes: put anything you like in any box, swap it for something of a completely different kind whenever you please, no questions asked. A numpy array is a row of boxes cast from a single mould, all the same size and shape, decided the moment the row was built. You can still change what's *in* a box — but whatever you put there gets squeezed to fit the mould, sometimes without so much as a warning.

In [8]:
# five velocity readings from a falling object, meant to be whole numbers of m/s
velocities = np.array([0, 10, 20, 29, 39])
print(velocities.dtype)     # int64 -- the mould was cast from integers

velocities[2] = 19.6        # the true value at that instant, from v = g*t
print(velocities)

int64
[ 0 10 19 29 39]


`19.6` went in; `19` came out. No error, no warning — the array's mould was set to whole numbers the moment it was created (`velocities.dtype` says `int64`), and every new value poured into it gets silently squeezed to fit that mould. Compare a plain Python list, which has no mould at all:

In [9]:
velocities_list = [0, 10, 20, 29, 39]
velocities_list[2] = 19.6
print(velocities_list)      # 19.6 survives exactly, because a list never forces one shared type

[0, 10, 19.6, 29, 39]


---
# 4. Recap

Three things worth carrying forward, none of them physics:

- **Dunders** (`__name__`, `__init__`, `__len__`, ...) are Python's own reserved vocabulary — read the ones the language already defines; don't invent new double-underscore names of your own.
- **`from module import *`** can silently replace Python's own builtins (`sum`, `abs`, `min`, `max`), and the replacement outlives the cell it happened in. Prefer `import numpy as np` and always write `np.` in front of numpy's functions.
- **numpy arrays are statically typed**; assigning a value that doesn't fit the array's dtype gets silently coerced, not rejected — check `.dtype` when a number looks wrong for no visible reason.

None of these will stop you from writing correct Python. What they *will* do, sooner or later, is cost you an afternoon each — this interlude is here so that afternoon is shorter.

— *Isht*

---

> #### 📌 Author's figure notes (for Isht — remove before sharing, or keep as a to-do)
> - **[FIGURE I.1]** — *The barcode and the dog's name.* Two labels on the same dog: a handwritten name tag (your variable names) and a vet-clinic barcode (`__name__` and friends). Caption: you choose one, Python owns the other.
> - **[FIGURE I.2]** — *The unlabelled workbench.* A labelled toolbox (`import math`, tools reached as `math.sqrt`) beside a workbench with tools dumped loose (`from math import *`), one of which — drawn identically to a tool already on the bench — is quietly sitting on top of it.
> - **[FIGURE I.3]** — *The mould.* A row of numbered boxes cast from one rigid mould (int64), with a value (`19.6`) being poured in and coming out visibly squeezed to fit (`19`), beside an ordinary row of open boxes where it goes in untouched.
>
> **[VERIFY before print:]** confirm on the actual target numpy version that `from numpy import *` still shadows `sum`/`abs`/`min`/`max` as described — this is standard numpy behaviour but worth a one-line sanity check before this goes to print.